# Hannah — real talking video (free test)

This makes **one test video** of Hannah saying the greeting, with free open-source models:
- **Kokoro-82M** (Apache-2.0) makes her voice
- **Ditto** (Apache-2.0, Ant Group) turns her picture + voice into a real talking video

**How to run (about 10–15 minutes):**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**
2. Menu **Runtime → Run all**
3. When step 3 shows a **Choose files** button, pick `hannah-plate.jpg`
   (in the project: `v2/public/nexora/hannah/hannah-plate.jpg`)
4. At the end, a zip downloads: `hannah_test.zip`. Send it back (or put it in the project folder).

Nothing here costs money. Your files stay in your own Colab session.

In [ ]:
# 1. Check the free GPU
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu or 'NO GPU! Use Runtime → Change runtime type → T4 GPU, then Runtime → Run all again.')

In [ ]:
# 2. Install everything (takes a few minutes)
!apt-get -qq -y install espeak-ng ffmpeg > /dev/null
!git clone -q --depth 1 https://github.com/antgroup/ditto-talkinghead /content/ditto
!pip install -q "kokoro>=0.9.4" soundfile librosa filetype imageio imageio-ffmpeg scikit-image cython tqdm mediapipe onnxruntime-gpu huggingface_hub
from huggingface_hub import snapshot_download
snapshot_download('digital-avatar/ditto-talkinghead', local_dir='/content/ditto/checkpoints',
                  allow_patterns=['ditto_pytorch/*', 'ditto_pytorch/**', 'ditto_cfg/*'])
!ls /content/ditto/checkpoints /content/ditto/checkpoints/ditto_pytorch
print('Installed.')

In [ ]:
# 3. Upload Hannah's picture (hannah-plate.jpg)
from google.colab import files
import shutil, os
up = files.upload()
name = list(up.keys())[0]
shutil.move(name, '/content/hannah.jpg')
print('Picture ready:', name)

In [ ]:
%%writefile /content/tts.py
# 4. Her voice: a small script that turns text into speech with Kokoro
import sys, numpy as np, soundfile as sf
from kokoro import KPipeline
text, voice, out = sys.argv[1], sys.argv[2], sys.argv[3]
pipe = KPipeline(lang_code='b' if voice.startswith('b') else 'a')
audio = np.concatenate([np.asarray(a) for _, _, a in pipe(text, voice=voice, speed=0.95)])
pad = np.zeros(int(24000 * 0.35), dtype=np.float32)          # calm start and end, for joining clips
sf.write(out, np.concatenate([pad, audio, pad]), 24000)
print('wrote', out)

In [ ]:
# 4b. Make the greeting, 3 voice samples to choose from, and silence for the idle loop
GREETING = "Hello and welcome to Nexora! I'm Hannah, your virtual receptionist. How can I help you today?"
!mkdir -p /content/out
!python /content/tts.py "$GREETING" af_heart /content/out/greeting.wav
for v in ['af_heart', 'af_bella', 'bf_emma']:
    !python /content/tts.py "Hi, I'm Hannah. Welcome to Nexora." {v} /content/out/voice_sample_{v}.wav
# 6 seconds of silence: she just sits, breathes and blinks (for the idle loop)
import numpy as np, soundfile as sf
sf.write('/content/out/silence.wav', np.zeros(16000 * 6, dtype=np.float32), 16000)

In [ ]:
# 5. Make the videos with Ditto (a few minutes)
%cd /content/ditto
!python inference.py --data_root ./checkpoints/ditto_pytorch --cfg_pkl ./checkpoints/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl --audio_path /content/out/greeting.wav --source_path /content/hannah.jpg --output_path /content/out/hannah_greeting.mp4
!python inference.py --data_root ./checkpoints/ditto_pytorch --cfg_pkl ./checkpoints/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl --audio_path /content/out/silence.wav --source_path /content/hannah.jpg --output_path /content/out/hannah_idle.mp4
!ls -la /content/out

In [ ]:
# 6. Watch it here, then download everything
from IPython.display import HTML
from base64 import b64encode
mp4 = open('/content/out/hannah_greeting.mp4', 'rb').read()
display(HTML('<video width=720 controls autoplay><source src="data:video/mp4;base64,' + b64encode(mp4).decode() + '"></video>'))
!cd /content/out && zip -q -r /content/hannah_test.zip .
files.download('/content/hannah_test.zip')